# 08 — Model comparison on the original labels (approach A, complete)

Pure ML predicting oil, water and gas **directly** from sensor features, with 9 model types × 2 feature sets.
Same leave-one-test-out validation as everything else.

| Model | Family | Idea |
|---|---|---|
| Ridge | linear | straight-line relation, penalised to avoid overfitting |
| LightGBM, XGBoost, CatBoost | gradient boosting | many small trees, each fixing the previous ones' errors |
| RandomForest, ExtraTrees | tree averaging | average of many trees on resampled data |
| GaussianProcess | kernel / Bayesian | prediction = weighted similarity to known tests + noise term |
| SVR | kernel | smooth function, ignores small errors |
| kNN | neighbours | average of the 3 most similar tests |

Feature sets: **raw** (sensor averages, choke, days) and **physics** (raw + choke-flow term, Gilbert term,
tubing gradient, Darcy J, ...). The well identity is given to every model as 10 one-hot columns (`is_GL-01` ...),
so all models see exactly the same inputs.

⏱ ~6–8 minutes.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..', 'src').resolve()))
import load
import validation as V
import vfm

tt = pd.read_parquet(load.PROCESSED / 'test_features.parquet')
feature_sets = {'raw': vfm.RAW_FEATURES, 'physics': vfm.PHYSICS_FEATURES}

results, predictions = [], {}
for model_name, make_model in vfm.MODELS.items():
    for set_name, features in feature_sets.items():
        name = f'A {model_name} {set_name} feats'
        method = vfm.make_ml_method(features, make_model, use_well_id=True, well_encoding='onehot')
        predictions[name], _ = V.run_experiment(name, tt, method, results)
        print(f'{name:40s} mean MAPE {results[-1]["mean_MAPE"]:.2f}', flush=True)

# does the well identity matter? (best boosting model without it)
name = 'A CatBoost physics feats NO well id'
predictions[name], _ = V.run_experiment(
    name, tt, vfm.make_ml_method(vfm.PHYSICS_FEATURES, vfm.catboost, use_well_id=False), results)

V.save_results(results, load.ROOT / 'outputs' / 'ablation_results.csv')
zoo = pd.DataFrame(results)[['experiment', 'oil_MAPE', 'water_MAPE', 'gas_MAPE', 'mean_MAPE']]
zoo.sort_values('mean_MAPE').round(2)

## Best ML model vs physics: paired significance test

In [ ]:
physics_pred = V.leave_one_test_out(tt, vfm.physics)
best = zoo.sort_values('mean_MAPE').experiment.iloc[0]
print('best ML model:', best)
V.paired_test(tt, physics_pred, predictions[best]).round(3)

In [ ]:
# Chart for the presentation: mean MAPE of every ML model vs the physics model and naive baselines
pivot = zoo.assign(model=zoo.experiment.str.replace('A ', '').str.replace(' feats', ''))
pivot = pivot[~pivot.model.str.contains('NO well')]
pivot['features'] = pivot.model.str.split().str[-1]
pivot['model'] = pivot.model.str.split().str[0]
table = pivot.pivot(index='model', columns='features', values='mean_MAPE').sort_values('physics')

ax = table.plot.barh(figsize=(9, 5))
ax.axvline(4.04, color='green', ls='--', label='physics model (4.04)')
ax.axvline(5.47, color='grey', ls=':', label='time interpolation (5.47)')
ax.set_xlabel('mean MAPE % (oil, water, gas) — lower is better')
ax.legend()
plt.tight_layout()
plt.show()
table.round(2)

**Conclusions**
- Best pure-ML model: **Gaussian Process** (physics features) — mean 4.26 % vs physics 4.04 %. Boosting models
  (LightGBM 8.1, XGBoost 8.8, CatBoost 6.2) and random forest are much worse: with 84 rows, trees overfit.
  Smooth models (GP, Ridge, kNN) do best -> the true relationships are smooth (as the physics says).
- Physics features help most models (Ridge 6.5 -> 4.9, CatBoost 7.2 -> 6.2). Well identity matters
  (CatBoost without it 6.2 -> 6.7).
- **GP vs physics, per target:** oil worse (2.81 vs 2.40, p=0.12), gas worse (6.71 vs 5.74, p=0.08), but
  **water significantly better: 3.24 vs 3.96 %, better on 55/85 tests, p = 0.002.**
  Caveat: we tried ~30 models, so one strong p-value can appear by luck; 0.002 still passes a strict
  correction (0.05 / 20 = 0.0025), but only just.
- -> Candidate final model: physics for oil and gas, physics + GP for water (to test next as an ensemble).
